# 🧠 Mental Health in Tech - Exploratory Data Analysis

This notebook performs comprehensive EDA on the Mental Health in Tech Survey dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded successfully!')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/raw/mental_health_survey.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Dataset Overview

In [ ]:
print('=== Dataset Info ===')
print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')
print(f'\n=== Data Types ===')
print(df.dtypes)
print(f'\n=== Statistical Summary ===')
df.describe()

## 3. Missing Values Analysis

In [ ]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Missing %', ascending=False)

missing = missing[missing['Missing Count'] > 0]
print('Columns with missing values:')
print(missing)

if len(missing) > 0:
    plt.figure(figsize=(10, 5))
    sns.barplot(x=missing.index, y=missing['Missing %'])
    plt.title('Missing Values by Column')
    plt.xticks(rotation=45)
    plt.ylabel('Missing %')
    plt.tight_layout()
    plt.savefig('../reports/missing_values.png', dpi=150)
    plt.show()

## 4. Class Distribution (Target Variable)

In [ ]:
print('=== Treatment Distribution ===')
print(df['treatment'].value_counts())
print(f'\nPercentage:')
print(df['treatment'].value_counts(normalize=True) * 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['treatment'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('Treatment Distribution')
axes[0].set_xlabel('Treatment')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

df['treatment'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                     colors=['steelblue', 'coral'])
axes[1].set_title('Treatment Distribution (%)')

plt.tight_layout()
plt.savefig('../reports/class_distribution.png', dpi=150)
plt.show()

## 5. Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

if 'Age' in df.columns:
    df['Age'].hist(bins=30, ax=axes[0, 0], color='steelblue', edgecolor='black')
    axes[0, 0].set_title('Age Distribution')
    axes[0, 0].set_xlabel('Age')

if 'Gender' in df.columns:
    df['Gender'].value_counts().plot(kind='bar', ax=axes[0, 1], color='coral')
    axes[0, 1].set_title('Gender Distribution')
    axes[0, 1].tick_params(axis='x', rotation=0)

if 'family_history' in df.columns:
    df['family_history'].value_counts().plot(kind='bar', ax=axes[1, 0], color='steelblue')
    axes[1, 0].set_title('Family History Distribution')
    axes[1, 0].tick_params(axis='x', rotation=0)

if 'work_interfere' in df.columns:
    df['work_interfere'].value_counts().plot(kind='bar', ax=axes[1, 1], color='coral')
    axes[1, 1].set_title('Work Interference Distribution')
    axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../reports/feature_distributions.png', dpi=150)
plt.show()

## 6. Correlation Analysis

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
for col in df_encoded.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

corr_matrix = df_encoded.corr()

plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=16)
plt.tight_layout()
plt.savefig('../reports/correlation_heatmap.png', dpi=150)
plt.show()

## 7. Target Analysis by Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

if 'family_history' in df.columns:
    pd.crosstab(df['family_history'], df['treatment']).plot(kind='bar', ax=axes[0, 0])
    axes[0, 0].set_title('Treatment by Family History')
    axes[0, 0].tick_params(axis='x', rotation=0)

if 'work_interfere' in df.columns:
    pd.crosstab(df['work_interfere'], df['treatment']).plot(kind='bar', ax=axes[0, 1])
    axes[0, 1].set_title('Treatment by Work Interference')
    axes[0, 1].tick_params(axis='x', rotation=0)

if 'benefits' in df.columns:
    pd.crosstab(df['benefits'], df['treatment']).plot(kind='bar', ax=axes[1, 0])
    axes[1, 0].set_title('Treatment by Benefits')
    axes[1, 0].tick_params(axis='x', rotation=0)

if 'care_options' in df.columns:
    pd.crosstab(df['care_options'], df['treatment']).plot(kind='bar', ax=axes[1, 1])
    axes[1, 1].set_title('Treatment by Care Options')
    axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../reports/target_analysis.png', dpi=150)
plt.show()

## 8. Outlier Detection

In [ ]:
if 'Age' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.boxplot(y=df['Age'], ax=axes[0], color='steelblue')
    axes[0].set_title('Age - Box Plot')

    sns.violinplot(y=df['Age'], ax=axes[1], color='coral')
    axes[1].set_title('Age - Violin Plot')

    plt.tight_layout()
    plt.savefig('../reports/outlier_detection.png', dpi=150)
    plt.show()

    Q1 = df['Age'].quantile(0.25)
    Q3 = df['Age'].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df['Age'] < Q1 - 1.5 * IQR) | (df['Age'] > Q3 + 1.5 * IQR)]
    print(f'Number of age outliers: {len(outliers)}')

## 9. Key Insights

1. **Class Balance**: The treatment target is relatively balanced.
2. **Family History**: Strong correlation with treatment seeking.
3. **Work Interference**: Higher interference correlates with treatment seeking.
4. **Benefits**: Having mental health benefits is associated with treatment.
5. **Age**: Most respondents are between 25-45 years old.